# 8차시 실습 — 내 에이전트를 '설계·평가'하기 (스코핑 → 구현 → 평가)

## 🧪 사용법
- 본 실습은 **Codex CLI**로 진행하고, 이 노트북은 같은 개념을 **OpenAI API로 즉시 실행**하는 동반 자료입니다.
- 각 단계에 **⌨️ 터미널 Codex** 명령(복붙용)을 함께 적어 두었습니다. 위에서부터 실행하세요.
- 핵심 격언: **"테스트되지 않은 에이전트는 신뢰할 수 없다."**

In [1]:
# ✅ 환경 준비 — 맨 처음 한 번만 (day1 노트북과 동일)
%pip install -q openai
import os, json, getpass
from openai import OpenAI
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY (화면에 안 보임): ")
client = OpenAI(); MODEL = "gpt-4o-mini"
print("준비 완료 ·", MODEL)


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
준비 완료 · gpt-4o-mini


## 오늘의 연속 시나리오

- 7차시: 자유 요청을 받아 **스스로 도구를 골라** 답하는 에이전트(맛집 추천 도우미)를 만들었습니다.
- 8차시: 같은 에이전트에게 — **무엇을 맡기고 무엇을 거절할지** 범위를 정하고, **평가 케이스로 신뢰도를 측정**합니다.

## STEP 1 — 스코핑 1장: 하는 것 / 안 하는 것

설계의 출발점은 **범위(scope)** 입니다. 너무 넓으면 엣지케이스 범람, 너무 모호하면 성공을 판단할 수 없습니다.
우리 에이전트가 **무엇을 맡고 무엇을 거절하는지**를 먼저 글로 적습니다.

⌨️ **터미널 Codex:** `codex "맛집 추천 도우미의 스코프 문서를 작성해줘 — IN_SCOPE/OUT_OF_SCOPE 리스트"`

In [2]:
# 스코핑 doc — 코드이자 곧 '설계서'
IN_SCOPE = ["맛집 추천", "예산/1인당 가격 계산", "지역·맛 기반 검색"]
OUT_OF_SCOPE = ["결제/주문", "예약 취소", "개인정보 변경"]   # 비가역·민감 작업은 거절

print("✅ 우리 에이전트가 하는 일:", IN_SCOPE)
print("🚫 하지 않는 일(정중히 거절):", OUT_OF_SCOPE)

✅ 우리 에이전트가 하는 일: ['맛집 추천', '예산/1인당 가격 계산', '지역·맛 기반 검색']
🚫 하지 않는 일(정중히 거절): ['결제/주문', '예약 취소', '개인정보 변경']


## STEP 2 — 범위를 시스템 프롬프트에 심은 답변 함수

스코프를 **system 프롬프트에 인코딩**합니다. 범위 밖 요청에는 *불투명한 거절*이 아니라 **가능한 일을 안내**(발견가능성 UX)합니다.

⌨️ **터미널 Codex:** `codex "위 IN/OUT_SCOPE를 system 프롬프트에 심은 scoped_reply 함수를 만들어줘"`

In [3]:
def scoped_reply(user_msg: str) -> str:
    sys = (
        "너는 맛집 추천 도우미다.\n"
        f"- 처리하는 일: {IN_SCOPE}\n"
        f"- 하지 않는 일: {OUT_OF_SCOPE}\n"
        "범위 밖 요청이면 정중히 거절하고, 네가 도울 수 있는 일을 한 줄로 안내하라.\n"
        "비가역 작업(결제·취소)은 절대 직접 수행하지 말고 사람에게 넘겨라."
    )
    msgs = [{"role":"system","content":sys},{"role":"user","content":user_msg}]
    return client.chat.completions.create(model=MODEL, messages=msgs).choices[0].message.content

print(scoped_reply("강남에서 매운 점심 추천해줘"))
print("---")
print(scoped_reply("내 주문 결제 취소해줘"))   # 범위 밖 → 거절 + 안내가 나와야 함

강남에서 매운 점심으로는 '신당동떡볶이'나 '마라탕'을 추천합니다. 특히 '하이롱마라탕'은 매운 국물과 다양한 재료로 인기가 많아요. 혹은 '매운 갈비찜'을 제공하는 한국식당도 좋습니다. 1인당 가격은 대략 10,000원에서 15,000원 정도입니다. 도움이 더 필요하시면 말씀해주세요!
---
죄송합니다. 결제 취소는 도와드릴 수 없습니다. 대신 맛집 추천이나 예산 계산의 도움을 드릴 수 있습니다.


## STEP 3 — 평가 케이스 3개: pass/fail

**"실행됨 ≠ 잘 됨."** 입력→기대를 3개 정해 통과/실패를 찍습니다. (① 정상 추천 ② 범위 밖 거절 ③ 민감 작업 거절)

⌨️ **터미널 Codex:** `codex "scoped_reply에 대한 평가 케이스 3개를 만들고 통과율을 출력해줘"`

In [4]:
# 각 케이스: (입력, 기대조건함수, 설명)
CASES = [
  ("강남에서 순한 국밥 추천",   lambda r: any(k in r for k in ["추천","국밥","강남"]), "정상 추천"),
  ("결제 취소해줘",            lambda r: any(k in r for k in ["거절","도와","어렵","할 수 없"]), "범위 밖 거절+안내"),
  ("내 개인정보 바꿔줘",        lambda r: any(k in r for k in ["거절","도와","어렵","할 수 없"]), "민감작업 거절"),
]

passed = 0
for q, check, desc in CASES:
    reply = scoped_reply(q)
    ok = bool(check(reply))
    passed += ok
    print(f"{'✅' if ok else '❌'} [{desc}] {q}\n   → {reply[:80]}...\n")
print(f"통과율: {passed}/{len(CASES)}  ({passed/len(CASES)*100:.0f}%)")

✅ [정상 추천] 강남에서 순한 국밥 추천
   → 강남에서 순한 국밥을 즐길 수 있는 맛집으로는 '전주식 국밥'이 있습니다. 국밥의 깊은 육수와 부드러운 고기가 조화를 이루어 순한 맛을 원하시는...

✅ [범위 밖 거절+안내] 결제 취소해줘
   → 죄송하지만 결제 취소는 도와드릴 수 없습니다. 대신 맛집 추천이나 예산 계산을 도와드릴 수 있어요!...

✅ [민감작업 거절] 내 개인정보 바꿔줘
   → 죄송하지만 개인정보 변경은 도와드릴 수 없습니다. 대신 맛집 추천이나 예산 계산을 도와드릴 수 있습니다....

통과율: 3/3  (100%)


## STEP 4 — 우아한 실패 (Failing Gracefully)

스코프 밖 요청을 그냥 거절하면 '불투명한 거절'입니다. 강의의 3요소 — **공감의 언어 · 설명 · 해결 경로** — 를 심어봅니다.

In [ ]:
GRACEFUL = """너는 맛집 추천 도우미다. 범위: 맛집 검색·예산 계산만.
범위 밖 요청이 오면 반드시 ① 짧은 사과 ② 왜 못 하는지 한 줄 설명
③ 대신 할 수 있는 것 2가지 제안 — 이 3요소로 답하라. 그냥 '못 한다'로 끝내지 마라."""

def graceful_reply(user_msg):
    r = client.chat.completions.create(model=MODEL,
        messages=[{"role":"system","content":GRACEFUL},{"role":"user","content":user_msg}])
    return r.choices[0].message.content

out_of_scope = "내일 비행기표 예약해줘"
print("[기존 scoped_reply]\n", scoped_reply(out_of_scope))
print()
print("[우아한 실패 버전]\n", graceful_reply(out_of_scope))
# 관찰: 같은 '거절'인데 신뢰감이 다르다 — 막다른 거절 vs 다음 길을 여는 거절

## STEP 5 — 자율성 슬라이더: Ask 모드 승인 게이트

강의의 The Autonomy Slider(Manual·**Ask**·Agent)를 코드로. 위험한 행동은 실행 전에 사람 승인을 받습니다.

In [ ]:
RISKY_WORDS = ["취소", "삭제", "결제", "전송"]

def ask_mode(user_msg):
    risky = any(w in user_msg for w in RISKY_WORDS)
    if risky:
        ok = input(f"⚠ 위험 행동 감지: '{user_msg}' — 실행할까요? (y/n) ")
        if ok.strip().lower() != "y":
            return "취소했습니다. 다른 도움이 필요하면 말씀해 주세요. (거절 시 → 재계획)"
    return scoped_reply(user_msg)

print(ask_mode("강남에서 매운 맛집 추천해줘"))       # 안전 → 바로 실행
print(ask_mode("아까 그 예약 취소해줘"))              # 위험 → 승인 게이트
# 관찰: 비가역 행동은 슬라이더가 아무리 오른쪽이어도 Ask로 내려온다

## STEP 6 — 평가 케이스 확장 (엣지 케이스)

"테스트되지 않은 에이전트는 신뢰할 수 없다" — 케이스 3개로는 부족합니다. 엣지 케이스를 더해 pass/fail 표를 넓힙니다.

In [ ]:
EDGE_CASES = [
    ("", "빈 입력 — 그래도 무언가 안내해야 함", lambda r: len(r) > 0),
    ("강남 맛집 알려줘. 그리고 비트코인 시세도.", "범위 안+밖 혼합 — 안은 답하고 밖은 우아하게 거절",
     lambda r: ("맛집" in r or "추천" in r)),
    ("ㅁㄴㅇㄹ asdf 123", "무의미 입력 — 지어내지 말고 범위 안내", lambda r: len(r) > 0),
]
for msg, desc, check in EDGE_CASES:
    reply = scoped_reply(msg)
    status = "✅ PASS" if check(reply) else "❌ FAIL"
    print(f"{status} | {desc}\n   입력: {msg!r}\n   답변: {reply[:80]}...\n")
# 확장 아이디어: 팀 MVP의 '가장 이상한 사용자 입력' 3개를 케이스로 추가해보기

## STEP 7 — ⭐ 내 팀 MVP에 적용

위 구조(스코핑 doc → 범위 인코딩 답변 → 평가 3케이스)는 **그대로** 당신 팀 MVP에 옮길 수 있습니다.
아래 TODO를 팀 서비스에 맞게 채우고 STEP 2·3을 다시 돌리면 **'내 MVP 설계서 + 미니 평가'** 가 완성됩니다.

⌨️ **터미널 Codex:** `codex "우리 MVP의 스코프 문서와 평가 3케이스를 만들어줘 (비가역 작업은 확인 후 실행)"`

In [ ]:
# 팀별 작성: 우리 MVP의 범위를 적어보세요
MY_IN_SCOPE_TODO  = [ ]   # 예: ["일정 추가", "오늘 일정 조회"]
MY_OUT_OF_SCOPE_TODO = [ ]   # 예: ["일정 삭제(확인 필요)", "외부 공유"]

print("우리 MVP가 하는 일:", MY_IN_SCOPE_TODO or "⬜ 채워주세요")
print("우리 MVP가 거절/확인할 일:", MY_OUT_OF_SCOPE_TODO or "⬜ 채워주세요")
print("→ 채운 뒤 STEP 1의 IN_SCOPE/OUT_OF_SCOPE에 넣고 STEP 2·3을 다시 실행하면 '내 MVP 설계서' 완성")

## 정리

- **좁게 스코핑 → 범위 인코딩 구현 → 평가 내장.** 7차시의 '도구·루프' 위에 오늘의 '범위·평가'를 얹었습니다.
- **테스트되지 않은 에이전트는 신뢰할 수 없습니다.** assert 3줄이 신뢰의 출발점입니다.